# SCOPE: Semantic COdec Pre-Editing for Object-Preserving Compression (ACCV 2026)

Notebook chạy thực nghiệm trên **Google Colab** cho kho mã nguồn đã đối chiếu thông số bài báo [`munnn01/scope_again`](https://github.com/munnn01/scope_again) (dựa trên bản chính thức [`miccunifi/SCOPE`](https://github.com/miccunifi/SCOPE), `arXiv:2610.39778`).

- Xem bảng đối chiếu chi tiết toàn bộ phương trình, kiến trúc và siêu tham số tại `docs/PAPER_VERIFICATION.md`.
- Chọn Runtime **GPU** (`Runtime -> Change runtime type -> T4 / L4 / A100 GPU`) trước khi chạy.

In [ ]:
# 1. Kiểm tra GPU, clone repo và cài đặt thư viện cho Google Colab
!nvidia-smi
!git clone https://github.com/munnn01/scope_again.git
%cd scope_again

# Giữ nguyên PyTorch CUDA có sẵn trên Colab, chỉ cài các gói phụ thuộc của SCOPE
!pip install -q -r requirements-colab.txt
!apt-get update -qq && apt-get install -y -qq libopenjp2-tools

In [ ]:
# (Tùy chọn) Cài Detectron2 khi cần huấn luyện từ đầu với giáo viên Faster R-CNN X101-FPN
# hoặc chấm điểm lại ảnh nén bằng Faster R-CNN:
# !pip install --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
# 2. Chạy bộ kiểm thử tự động & kiểm tra cấu hình thông số bài báo
!python tools/run_full_tests.py
!python tools/validate_environment.py --mode static --show-ok

In [ ]:
# 3. Tải trọng số SCOPE FULL chính thức từ Google Drive của tác giả
import os
ckpt_dir = 'runs/preedit_d2fpn_nojp2_unbiased_linearsev_mediumbpg_sevbasis_scratch_FULL_20260701_140557'
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path = f'{ckpt_dir}/best_val.pt'
if not os.path.exists(ckpt_path):
    !gdown --id 1M7lHWbNjrckMNBC623Cn5PvBDuDUBlda -O {ckpt_path}

!python tools/check_runtime.py --checkpoints

In [ ]:
# 4. Demo nhanh tiền xử lý SCOPE + nén JPEG thực tế trên 1 ảnh mẫu COCO
import urllib.request
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from scope.inference import load_scope_model
from scope.evaluation.utils import jpeg_encode_decode

sample_url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
urllib.request.urlretrieve(sample_url, 'sample_coco.jpg')
img_bgr = cv2.imread('sample_coco.jpg')
h, w = img_bgr.shape[:2]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, fam_map = load_scope_model(ckpt_path, device)

jpeg_q = 20
severity = (99.0 - jpeg_q) / 98.0  # Chuẩn hóa tuyến tính đúng bài báo: q in [1, 99]
x = torch.from_numpy(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
with torch.no_grad():
    edited, delta_obj, delta_bg, attn, _ = model(
        x,
        severity=torch.tensor([severity], device=device),
        codec_family=torch.tensor([fam_map['diff_jpeg']], device=device),
    )

edited_bgr = cv2.cvtColor((edited[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8), cv2.COLOR_RGB2BGR)
base_rec, base_bytes = jpeg_encode_decode(img_bgr, quality=jpeg_q)
scope_rec, scope_bytes = jpeg_encode_decode(edited_bgr, quality=jpeg_q)

base_bpp = base_bytes * 8.0 / (h * w)
scope_bpp = scope_bytes * 8.0 / (h * w)
print(f'JPEG q={jpeg_q} | Direct: {base_bpp:.4f} bpp ({base_bytes} B) | SCOPE: {scope_bpp:.4f} bpp ({scope_bytes} B) | Rate delta: {(scope_bpp/base_bpp - 1)*100:+.2f}%')

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original')
axes[1].imshow(attn[0, 0].cpu().numpy(), cmap='inferno', vmin=0, vmax=1); axes[1].set_title(f'Attention A (s={severity:.2f})')
axes[2].imshow(cv2.cvtColor(base_rec, cv2.COLOR_BGR2RGB)); axes[2].set_title(f'Direct JPEG q={jpeg_q} ({base_bpp:.3f} bpp)')
axes[3].imshow(cv2.cvtColor(scope_rec, cv2.COLOR_BGR2RGB)); axes[3].set_title(f'SCOPE + JPEG q={jpeg_q} ({scope_bpp:.3f} bpp)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 5. (Tùy chọn) Tải COCO 2017 val2017 + YOLO26x để chạy đánh giá đường cong Rate-mAP
# !mkdir -p datasets/coco-2017/validation models
# !wget -q http://images.cocodataset.org/zips/val2017.zip
# !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
# !unzip -q val2017.zip && mv val2017 datasets/coco-2017/validation/data
# !unzip -q annotations_trainval2017.zip && mv annotations/instances_val2017.json datasets/coco-2017/validation/labels.json
# !python -c "from ultralytics import YOLO, RTDETR; YOLO('yolo26x.pt'); RTDETR('rtdetr-l.pt')" && mv yolo26x.pt rtdetr-l.pt models/

# Chạy thử nghiệm nhanh trên tập con 20 ảnh cho JPEG & BMSHJ:
# !python tools/smoke_real.py --config configs/eval.yaml --limit 20 --experiments jpeg_q10,scope_jpeg_q10,bmshj_q1,scope_bmshj_q1

In [ ]:
# 6. (Tùy chọn) Huấn luyện SCOPE từ đầu theo đúng thông số bài báo (cần COCO train2017 + Detectron2)
# !bash training/train_scope.sh